# Repeated train/test-split LASSO models of CAR-T response

Monte Carlo cross-validation (101 stratified 70/30 splits; in each split an L1 logistic
regression is tuned by 5x5 repeated stratified CV on the training set and scored on the
test set). This notebook drives every repeated-CV model of the paper from one run table.

**Inputs** (`input_dir`): `counts/counts_norm_cart_paper.tsv.gz`, `metadata/sample_metadata.tsv`
(via `load_modeling_data`), gene sets from `utils/genesets.yaml`.

**Outputs** (`results/ml/`), each folder with `classifier_scores.csv` and `selected_features.csv`:

| folder | model | used in |
|---|---|---|
| `{all,dlbcl,12mo}/{LN,FMAC,TEX}` | LymphoMAP archetype gene sets | Fig. 3, SFig. 3B/C, SFig. 17C |
| `{all,dlbcl,12mo}/LASSO` | all expressed genes (`filter_75`) | Fig. 3, Fig. 4 |
| `{all,dlbcl,12mo}/LASSO_SHUFFLE` | `filter_75` + shuffled labels (null control) | Fig. 3 |
| `{all,dlbcl,12mo}/RAND_30genes` | `filter_75` + 30 random genes (control) | Fig. 3 |
| `clinical_vars/<var>` | one clinical covariate, median-impute -> scale -> L1 (all samples) | SFig. 10A |
| `forced_clin_ridge/LN_<var>` | out-of-fold LN score + clinical covariate(s), L2 (ridge) final model (all samples) | SFig. 10B |

Cohorts: `all` = all 91 pre-infusion samples; `dlbcl` = de novo DLBCL (n = 67);
`12mo` = the >=12-month-outcome cohort (n = 85).

**Runtime.** Results are written to `results/ml/<run>/`. The full table takes roughly 3-4 hours
with `N_JOBS = 16` (the genome-wide `LASSO` and `LASSO_SHUFFLE` runs take about 20-30 min each per
cohort; each gene-set run about 5 min; the clinical, LN+clinical and random-gene runs a few minutes
each). With `SKIP_EXISTING = True` (default), runs whose output is already present in `results/ml/`
are skipped, so an interrupted run can be resumed. To rerun only some models, list them in
`RUNS_TO_EXECUTE` (e.g. `["all/LN", "dlbcl/TEX"]`) and set `SKIP_EXISTING = False` or delete their folders.

In [ ]:
import os
import sys
import time

sys.path.insert(0, "../../utils")
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from config import res_path
from data_loading import CLINICAL_VARS, build_clinical_matrix, load_modeling_data
from genesets import FMAC_ENSEMBL, LN_ENSEMBL, TEX_ENSEMBL
from modeling import (run_lasso_iterations, run_lasso_iterations_clinical,
                      run_lasso_iterations_LN_combined, save_results)

## Settings

In [ ]:
N_ITER = 101          # train/test splits per model (iteration i uses seed i)
N_JOBS = 16           # GridSearchCV workers (no effect on the results)

RUNS_TO_EXECUTE = None   # None = every run in the table below, or a list of run names
SKIP_EXISTING = True     # skip runs whose classifier_scores.csv already exists

# Output folder inside results_dir. The environment variable lets a full rerun be written
# to another folder (e.g. CFRNA_ML_DIR=ml_rerun) without overwriting existing results.
ML_DIR = os.environ.get("CFRNA_ML_DIR", "ml")

## Run table

One row per model: cohort x feature set x options -> output folder `results/ml/<run>`.

In [ ]:
GENE_SET_RUNS = {                       # name -> run_lasso_iterations options
    "LN":            dict(genes=LN_ENSEMBL),
    "FMAC":          dict(genes=FMAC_ENSEMBL),
    "TEX":           dict(genes=TEX_ENSEMBL),
    "LASSO":         dict(filter_75=True),
    "LASSO_SHUFFLE": dict(filter_75=True, shuffle_labels=True),
    "RAND_30genes":  dict(filter_75=True, random_select30=True),
}

RUNS = []
for cohort in ["all", "dlbcl", "12mo"]:
    for name, opts in GENE_SET_RUNS.items():
        RUNS.append(dict(run=f"{cohort}/{name}", kind="genes", cohort=cohort, **opts))
    if cohort == "all":
        # Clinical covariates alone, and LN score + clinical covariate(s) (ridge), all samples only.
        for var in CLINICAL_VARS:
            RUNS.append(dict(run=f"clinical_vars/{var}", kind="clinical", cohort=cohort, var=var))
        for var in CLINICAL_VARS + ["clinical_all"]:
            RUNS.append(dict(run=f"forced_clin_ridge/LN_{var}", kind="LN_clinical", cohort=cohort, var=var))

run_table = pd.DataFrame(RUNS).drop(columns="genes")
run_table["n_genes"] = [len(r["genes"]) if "genes" in r else np.nan for r in RUNS]
run_table

## Run the models

In [ ]:
DATA = {c: load_modeling_data(c) for c in ["all", "dlbcl", "12mo"]}   # (X, y, metadata)
for c, (X, y, meta) in DATA.items():
    print(f"{c:6s} n = {len(y)} ({(y == 'R').sum()} R / {(y == 'NR').sum()} NR), {X.shape[1]} genes")


def gene_subset(X, genes):
    return X.loc[:, X.columns.intersection(genes)]


def run_model(spec):
    X, y, meta = DATA[spec["cohort"]]
    strat = meta["stratify_group"]
    if spec["kind"] == "genes":
        X_run = gene_subset(X, spec["genes"]) if "genes" in spec else X
        return run_lasso_iterations(X_run, y, strat, n_iter=N_ITER, n_jobs=N_JOBS,
                                    filter_75=spec.get("filter_75", False),
                                    shuffle_labels=spec.get("shuffle_labels", False),
                                    random_select30=spec.get("random_select30", False))
    X_clin = build_clinical_matrix(meta, spec["var"])
    if spec["kind"] == "clinical":
        return run_lasso_iterations_clinical(X_clin, y, strat, n_iter=N_ITER, n_jobs=N_JOBS)
    if spec["kind"] == "LN_clinical":
        return run_lasso_iterations_LN_combined(X_clin, y, strat, X_LN=gene_subset(X, LN_ENSEMBL),
                                                n_iter=N_ITER, final_penalty="l2", n_jobs=N_JOBS)
    raise ValueError(spec["kind"])


for spec in RUNS:
    if RUNS_TO_EXECUTE is not None and spec["run"] not in RUNS_TO_EXECUTE:
        continue
    outdir = res_path(ML_DIR, spec["run"])
    if SKIP_EXISTING and (outdir / "classifier_scores.csv").exists():
        print(f"[skip] {spec['run']} (output exists)")
        continue
    print(f"[run]  {spec['run']}")
    t0 = time.time()
    res = run_model(spec)
    save_results(res, outdir)
    print(f"       saved to {outdir} ({(time.time() - t0) / 60:.1f} min)", flush=True)

## Summary: test AUC per model (from the saved outputs)

In [ ]:
def auc_per_iteration(run, y):
    scores = pd.read_csv(res_path(ML_DIR, run, "classifier_scores.csv"))
    test = scores[scores.train_or_test == "test"]
    return test.groupby("iteration").apply(
        lambda d: roc_auc_score(y.loc[d.sample_id], d.classifier_score))


rows = []
for spec in RUNS:
    if not res_path(ML_DIR, spec["run"], "classifier_scores.csv").exists():
        continue
    aucs = auc_per_iteration(spec["run"], DATA[spec["cohort"]][1])
    rows.append(dict(run=spec["run"], n_iter=len(aucs), median_test_auc=aucs.median(),
                     q25=aucs.quantile(0.25), q75=aucs.quantile(0.75)))
pd.DataFrame(rows).round(3)